In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['PruebaProyecto'] 
coleccion = db['G_2_Prueba'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [5]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print("🚀 Iniciando prueba de extracción Avanzada (con Scroll y Filtros)...")

# 1. Configuración Headless (Para entornos de servidor/Docker/Jupyter)
options = Options()
options.add_argument("--headless=new") # Modo invisible obligatorio para tu entorno
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

try:
    # 2. Navegación
    url = "https://www.portalinmobiliario.com/arriendo/departamento/coquimbo-coquimbo"
    print(f"🌐 Navegando a: {url}")
    driver.get(url)

    # 3. Esperar resultados iniciales
    print("⏳ Esperando carga inicial...")
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ui-search-layout__item"))
    )
    
    # --- LA MAGIA DEL SCROLL ---
    print("📜 Deslizando hasta el final de la página para cargar todo...")
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5) # Le damos 5 segundos para que carguen las imágenes y textos rezagados

    # 4. Capturar bloques
    bloques = driver.find_elements(By.CSS_SELECTOR, ".ui-search-layout__item")
    print(f"✅ Se detectaron {len(bloques)} bloques en total (incluyendo posibles anuncios).\n")
    print("-" * 50)

    # 5. Iterar sobre los primeros 10 bloques para ver si el filtro funciona
    propiedades_validas = 0
    
    for bloque in bloques[:10]:
        # Si ya encontramos 5 propiedades válidas, paramos la prueba
        if propiedades_validas >= 5:
            break
            
        try:
            # Extraemos los textos primero
            nombre = bloque.find_element(By.CSS_SELECTOR, ".poly-component__title").text
            precio_texto = bloque.find_element(By.CSS_SELECTOR, ".poly-component__price").text
            
            # --- FILTRO ANTI-BLANCOS ---
            # Si el nombre o el precio están vacíos, es publicidad o cargó mal. Lo ignoramos.
            if not nombre.strip() or not precio_texto.strip():
                print("   ⚠️ [Bloque ignorado: Publicidad o datos vacíos]")
                continue
                
            # Si pasamos el filtro, es una propiedad real
            propiedades_validas += 1
            print(f"🏠 PROPIEDAD VÁLIDA #{propiedades_validas}:")
            print(f"   ➤ Título: {nombre}")
            print(f"   ➤ Precio bruto: {precio_texto}")
            
            # Prueba de limpieza
            precio_limpio = precio_texto.replace("$", "").replace(".", "").strip()
            if precio_limpio.isdigit():
                print(f"   ➤ Precio limpio (para BD): {float(precio_limpio)}")
            else:
                print(f"   ➤ Precio limpio (para BD): 0.0 (No numérico)")
                
            print("-" * 50)
            
        except Exception:
            # Si ni siquiera tiene la clase del título, definitivamente es publicidad
            print("   ⚠️ [Bloque ignorado: Estructura no coincide]")
            continue

except Exception as e:
    print(f"🚨 Error crítico durante la prueba: {e}")

finally:
    # 6. Cierre seguro
    print("🧹 Prueba finalizada. Cerrando el navegador.")
    driver.quit()

🚀 Iniciando prueba de extracción Avanzada (con Scroll y Filtros)...
🌐 Navegando a: https://www.portalinmobiliario.com/arriendo/departamento/coquimbo-coquimbo
⏳ Esperando carga inicial...
📜 Deslizando hasta el final de la página para cargar todo...
✅ Se detectaron 48 bloques en total (incluyendo posibles anuncios).

--------------------------------------------------
🏠 PROPIEDAD VÁLIDA #1:
   ➤ Título: Arriendo Centro De Coquimbo: Depto Amoblado Con Vista Al Mar
   ➤ Precio bruto: $
450.000
   ➤ Precio limpio (para BD): 450000.0
--------------------------------------------------
🏠 PROPIEDAD VÁLIDA #2:
   ➤ Título: Se Arrienda Departamento Marzo A Diciembre Costa Elqui
   ➤ Precio bruto: $
650.000
   ➤ Precio limpio (para BD): 650000.0
--------------------------------------------------
🏠 PROPIEDAD VÁLIDA #3:
   ➤ Título: Arriendo Hermoso Penthouse A Pasos De Av Mar (148150)
   ➤ Precio bruto: $
1.500.000
   ➤ Precio limpio (para BD): 1500000.0
---------------------------------------------

In [21]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 1. FORZAR LA PANTALLA PARA noVNC
# Esto le dice a Chrome en qué monitor virtual debe dibujarse para que VNC lo pueda grabar
os.environ["DISPLAY"] = ":99"  # IMPORTANTE: Si sigue negra, cambia ":0" por ":1" o ":99"

# Cierra procesos viejos que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("Limpieza completada. Pantalla virtual configurada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "G2"
datos_finales = []   
driver = None        

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()

# 🚨 ¡ATENCIÓN! Comentamos el "--headless=new" para que puedas verlo en noVNC 🚨
# options.add_argument("--headless=new")

# Argumentos de estabilidad obligatorios para entornos Docker/Servidor
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    # Inicia el navegador
    driver = webdriver.Chrome(options=options)
    print("Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN ---
    limite_paginas = 3
    driver.get("https://www.portalinmobiliario.com/arriendo/departamento/coquimbo-coquimbo")

    for nivel_pagina in range(limite_paginas):
        print(f"\n--- Procesando Página {nivel_pagina + 1} ---")

        # Espera explícita a que existan resultados
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ui-search-layout__item"))
        )
        
        # --- SCROLL HUMANO (para que pueda tomar todos los datos ---
        for _ in range(5):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) # Pausa final para que asienten los últimos bloques

        # Capturamos todos los bloques presentes en la página
        bloques = driver.find_elements(By.CSS_SELECTOR, ".ui-search-layout__item")
        
        propiedades_extraidas = 0

        for i, bloque in enumerate(bloques):
            try:
                # --- LA MAGIA DEL textContent ---
                # Ahora extraemos el texto desde las entrañas del HTML, 
                # así no importa si la propiedad ya no está visible en pantalla
                nombre = bloque.find_element(By.CSS_SELECTOR, ".poly-component__title").get_attribute("textContent")
                try:
                    precio_texto = bloque.find_element(By.CSS_SELECTOR, ".poly-price__current").get_attribute("textContent")
                except:
                    precio_texto = bloque.find_element(By.CSS_SELECTOR, ".poly-component__price").get_attribute("textContent")

                # Filtro anti-blancos (por si acaso detecta un banner publicitario real)
                if not nombre or not precio_texto or not nombre.strip() or not precio_texto.strip():
                    continue

                try:
                    caracteristicas = bloque.find_element(By.CSS_SELECTOR, ".poly-component__attributes-list").get_attribute("textContent")

                except:
                    caracteristicas = "Sin datos"

                try:
                    direccion = bloque.find_element(By.CSS_SELECTOR, ".poly-component__location").get_attribute("textContent")

                except:
                    direccion = "Dirección no especificada"

                datos_finales.append({
                    "identificador": nombre.strip(),
                    "valor": precio_texto,
                    "caracteristicas": caracteristicas.strip(),
                    "direccion": direccion.strip(),
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
                propiedades_extraidas += 1
                
            except:
                # Si el bloque tiene otro HTML (Proyectos nuevos), lo ignora
                continue
                
        print(f"Se extrajeron {propiedades_extraidas} propiedades reales de {len(bloques)} bloques.")

        # Intenta avanzar a la siguiente página
        try:
            btn_sig = driver.find_element(By.CSS_SELECTOR, ".andes-pagination__button--next a")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
        except:
            print("No se encontró el botón siguiente o ya es la última página.")
            break

    print(f"\n Extracción terminada: {len(datos_finales)} propiedades obtenidas en total.")

except Exception as e:
    print(f"Error en Selenium: {e}")

finally:
    # Cierra el navegador
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    if datos_finales:
        client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
        db = client["PruebaProyecto"]
        coleccion = db["G_2_Prueba"] 

        for d in datos_finales:
            # Limpia el valor antes de convertirlo (Quitamos $, puntos y saltos de línea)
            v_limpio = str(d["valor"]).replace("$", "").replace(".", "").replace(",", "").replace("\n", "").strip()
            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0

        coleccion.insert_many(datos_finales)
        print("Datos cargados en MongoDB correctamente.")
    else:
        print("No hay datos válidos para guardar.")

except Exception as e:
    print(f"Error en MongoDB: {e}")

Limpieza completada. Pantalla virtual configurada.
Navegador iniciado correctamente.

--- Procesando Página 1 ---
Se extrajeron 48 propiedades reales de 48 bloques.

--- Procesando Página 2 ---
Se extrajeron 48 propiedades reales de 48 bloques.

--- Procesando Página 3 ---
Se extrajeron 48 propiedades reales de 48 bloques.

 Extracción terminada: 144 propiedades obtenidas en total.
Datos cargados en MongoDB correctamente.


In [22]:
from pyspark.sql import SparkSession

# Forzamos cierre de sesiones previas con versiones erróneas
try: spark.stop()
except: pass

spark = SparkSession.builder \
    .appName("Analisis_Prueba") \
    .config("spark.mongodb.read.connection.uri", "mongodb://mongodb:27017/PruebaProyecto.G_2_Prueba") \
    .getOrCreate()

df = spark.read.format("mongodb").load()
print(f"Éxito total: {df.count()} registros.")
df.show(5)

Éxito total: 144 registros.
+--------------------+--------------------+--------------------+-------------------+-----+--------------------+---------+
|                 _id|     caracteristicas|           direccion|      fecha_captura|grupo|       identificador|    valor|
+--------------------+--------------------+--------------------+-------------------+-----+--------------------+---------+
|69e2e3ca12d0c3e10...|1 dormitorio1 bañ...|Melgarejo 902, Ce...|2026-04-18 01:50:38|   G2|Arriendo Centro D...| 450000.0|
|69e2e3ca12d0c3e10...|2 dormitorios2 ba...|Peñuelas Norte, 1...|2026-04-18 01:50:40|   G2|Se Arrienda Depar...| 650000.0|
|69e2e3ca12d0c3e10...|4 dormitorios3 ba...|Avenida Los Pesca...|2026-04-18 01:50:41|   G2|Arriendo Hermoso ...|1500000.0|
|69e2e3ca12d0c3e10...|2 dormitorios2 ba...|Sindempart, Coquimbo|2026-04-18 01:50:44|   G2|Arriendo Departam...| 650000.0|
|69e2e3ca12d0c3e10...|2 dormitorios2 ba...|Avda Los Pescador...|2026-04-18 01:50:44|   G2|Arriendo Departam...| 630000

In [27]:
from pyspark.sql.functions import col, regexp_extract, avg, count, round, format_number, concat, lit

# para separar las caracteristicas
df_limpio = df.withColumn("dormitorios", regexp_extract(col("caracteristicas"), r"(\d+)\s*dorm",1).cast("int")) \
              .withColumn("baños", regexp_extract(col("caracteristicas"), r"(\d+)\s*baño",1).cast("int")) \
              .withColumn("metros_cuadrados", regexp_extract(col("caracteristicas"), r"(\d+)\s*m²",1).cast("float"))
df_limpio = df_limpio.na.fill({"dormitorios":0,"baños":0,"metros_cuadrados":0})

df_validado = df_limpio.filter(col("valor") > 10000)

print("Reporte por dormitorios")
reporte_dorms = df_validado.groupBy("dormitorios").agg(
        count("*").alias("cantidad_propiedades"),
        avg("valor").alias("precio_prom")
).withColumn(
     "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"),0),0))
).orderBy("dormitorios")

reporte_dorms.select("dormitorios","cantidad_propiedades","precio_promedio").show()

print("Reporte por baños")
reporte_baños = df_validado.groupBy("baños").agg(
        count("*").alias("cantidad_propiedades"),
        avg("valor").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"),0),0))
).orderBy("baños")

reporte_baños.select("baños","cantidad_propiedades","precio_promedio").show()

Reporte por dormitorios
+-----------+--------------------+---------------+
|dormitorios|cantidad_propiedades|precio_promedio|
+-----------+--------------------+---------------+
|          1|                  11|       $431,818|
|          2|                  75|       $572,800|
|          3|                  50|       $656,000|
|          4|                   5|     $1,530,000|
+-----------+--------------------+---------------+

Reporte por baños
+-----+--------------------+---------------+
|baños|cantidad_propiedades|precio_promedio|
+-----+--------------------+---------------+
|    1|                  58|       $494,828|
|    2|                  78|       $664,231|
|    3|                   4|     $1,600,000|
|    4|                   1|     $1,250,000|
+-----+--------------------+---------------+



In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['Prueba'] 
coleccion = db['Pruebaaa'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [5]:
# ==============================================================================
# PROYECTO BIG DATA - GRUPO 2 (REAL ESTATE)
# Módulo de Extracción: Portal Inmobiliario (Coquimbo)
# ==============================================================================

# --- PASO 0: LIMPIEZA Y PREPARACIÓN DEL ENTORNO ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Forzamos la pantalla virtual para noVNC en Docker
os.environ["DISPLAY"] = ":99"  

# Matamos procesos fantasmas para evitar que se coma la RAM
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza de memoria completada. Entorno virtual listo.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "G2"
datos_finales = []   
driver = None        

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
# options.add_argument("--headless=new") # Comentado para que veas el bot trabajar
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    driver = webdriver.Chrome(options=options)
    print("🚀 Navegador iniciado. Conectando a Portal Inmobiliario...")

    # --- PASO 2: NAVEGACIÓN Y SCROLL ---
    limite_paginas = 2 # 👈 Cambia esto a 15 cuando quieras ir por los 500 registros
    driver.get("https://www.portalinmobiliario.com/arriendo/departamento/coquimbo-coquimbo")

    for nivel_pagina in range(limite_paginas):
        print(f"\n--- 📄 Procesando Página {nivel_pagina + 1} ---")

        # Esperamos a que los departamentos carguen
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ui-search-layout__item"))
        )
        
        # Scroll Humano (Activa el Lazy Loading)
        print("Caminando por el pasillo virtual (haciendo scroll)...")
        for _ in range(6):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) 

        # --- PASO 3: EXTRACCIÓN DE DATOS ---
        bloques = driver.find_elements(By.CSS_SELECTOR, ".ui-search-layout__item")
        propiedades_extraidas = 0

        for i, bloque in enumerate(bloques):
            try:
                # 1. Título
                nombre = bloque.find_element(By.CSS_SELECTOR, ".poly-component__title").get_attribute("textContent")
                
                # 2. URL (Llave de Unión)
                try:
                    url = bloque.find_element(By.TAG_NAME, "a").get_attribute("href")
                    url = url.split("#")[0].split("?")[0] # Limpiamos rastreadores de MercadoLibre
                except:
                    url = "Sin URL"

                # 3. Precio Actual (Evitando el precio tachado)
                try:
                    precio_texto = bloque.find_element(By.CSS_SELECTOR, ".poly-price__current").get_attribute("textContent")
                except:
                    precio_texto = bloque.find_element(By.CSS_SELECTOR, ".poly-component__price").get_attribute("textContent")

                # Si es un anuncio vacío o publicidad, lo saltamos
                if not nombre or not precio_texto or not nombre.strip() or url == "Sin URL":
                    continue

                # 4. Características y Dirección (Tu aporte al equipo)
                try:
                    caracteristicas = bloque.find_element(By.CSS_SELECTOR, ".poly-component__attributes-list").get_attribute("textContent")
                except:
                    caracteristicas = "Sin datos"

                try:
                    direccion = bloque.find_element(By.CSS_SELECTOR, ".poly-component__location").get_attribute("textContent")
                except:
                    direccion = "Dirección no especificada"

                # 5. Empaquetado en el Diccionario
                datos_finales.append({
                    "url": url,
                    "identificador": nombre.strip(),
                    "valor": precio_texto,
                    "caracteristicas": caracteristicas.strip(),
                    "direccion": direccion.strip(),
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
                propiedades_extraidas += 1
                
            except:
                continue # Pasa al siguiente si hay un error en este bloque específico
                
        print(f"✅ Se extrajeron {propiedades_extraidas} propiedades reales en esta página.")

        # Avanzamos a la siguiente página
        try:
            btn_sig = driver.find_element(By.CSS_SELECTOR, ".andes-pagination__button--next a")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
        except:
            print("🛑 Fin de resultados en la web.")
            break

except Exception as e:
    print(f"🚨 Error crítico en Selenium: {e}")

finally:
    if driver is not None:
        try:
            driver.quit()
            print("Navegador cerrado.")
        except:
            pass

# --- PASO 4: GUARDADO EN MONGODB LOCAL (CON UPSERT) ---
print("\n--- 💾 Iniciando proceso de guardado ---")
try:
    if datos_finales:
        # Conexión al Docker Local (Cambiar esta URI cuando pasen a Atlas)
        client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
        db = client["Prueba"]
        coleccion = db["Pruebaaa"] 

        registros_nuevos = 0
        registros_actualizados = 0

        for d in datos_finales:
            # Limpiamos el texto del precio para guardarlo como número puro
            v_limpio = str(d["valor"]).replace("$", "").replace(".", "").replace(",", "").replace("\n", "").strip()
            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0

            # Lógica de Actualización Inteligente
            resultado = coleccion.update_one(
                {"url": d["url"]}, 
                {"$set": d},       
                upsert=True        
            )

            if resultado.upserted_id:
                registros_nuevos += 1
            else:
                registros_actualizados += 1

        print(f"🎉 ¡Misión cumplida!")
        print(f"   -> {registros_nuevos} departamentos nuevos guardados.")
        print(f"   -> {registros_actualizados} departamentos actualizados (Sin duplicados).")
    else:
        print("⚠️ No se encontró ningún dato para guardar.")

except Exception as e:
    print(f"🚨 Error conectando a la Base de Datos: {e}")

🧹 Limpieza de memoria completada. Entorno virtual listo.
🚀 Navegador iniciado. Conectando a Portal Inmobiliario...

--- 📄 Procesando Página 1 ---
Caminando por el pasillo virtual (haciendo scroll)...
✅ Se extrajeron 48 propiedades reales en esta página.

--- 📄 Procesando Página 2 ---
Caminando por el pasillo virtual (haciendo scroll)...
✅ Se extrajeron 48 propiedades reales en esta página.
Navegador cerrado.

--- 💾 Iniciando proceso de guardado ---
🎉 ¡Misión cumplida!
   -> 0 departamentos nuevos guardados.
   -> 96 departamentos actualizados (Sin duplicados).


In [5]:
# ==============================================================================
# PROYECTO BIG DATA - SCRAPING POR LOTES (GRUPAL)
# ==============================================================================

import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- CONFIGURACIÓN DEL LOTE (MODIFICA ESTO CADA VEZ QUE EJECUTES) ---
PAGINA_INICIAL = 2  # Cambia a 5, 9, 13... para las siguientes tandas
CUANTAS_PAGINAS = 3 # Procesaremos 4 páginas por ejecución (~192 registros)
# --------------------------------------------------------------------

os.environ["DISPLAY"] = ":99"
os.system("pkill -9 chrome && pkill -9 chromedriver")

options = Options()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

try:
    driver = webdriver.Chrome(options=options)
    
    # 1. Cálculo de la URL de inicio según la página deseada
    # Página 1 -> Desde_1, Página 2 -> Desde_49, etc.
    desde = 1 if PAGINA_INICIAL == 1 else ((PAGINA_INICIAL - 1) * 48) + 1
    url_base = f"https://www.portalinmobiliario.com/arriendo/departamento/coquimbo-coquimbo_Desde_{desde}_NoIndex_True"
    
    driver.get(url_base)
    print(f"🚀 Iniciando lote: Página {PAGINA_INICIAL} hasta {PAGINA_INICIAL + CUANTAS_PAGINAS - 1}")

    for i in range(CUANTAS_PAGINAS):
        pagina_actual = PAGINA_INICIAL + i
        print(f"\n--- 📄 Extrayendo Página {pagina_actual} ---")

        # Espera y Scroll Humano
        WebDriverWait(driver, 20).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ui-search-layout__item")))
        for _ in range(5):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(1)
        
        bloques = driver.find_elements(By.CSS_SELECTOR, ".ui-search-layout__item")
        datos_lote = []

        for bloque in bloques:
            try:
                # Extracción de etiquetas obligatorias y específicas [cite: 61, 68]
                nombre = bloque.find_element(By.CSS_SELECTOR, ".poly-component__title").get_attribute("textContent")
                url = bloque.find_element(By.TAG_NAME, "a").get_attribute("href").split("#")[0].split("?")[0]
                
                try:
                    p_text = bloque.find_element(By.CSS_SELECTOR, ".poly-price__current").get_attribute("textContent")
                except:
                    p_text = bloque.find_element(By.CSS_SELECTOR, ".poly-component__price").get_attribute("textContent")

                caract = bloque.find_element(By.CSS_SELECTOR, ".poly-component__attributes-list").get_attribute("textContent")
                loc = bloque.find_element(By.CSS_SELECTOR, ".poly-component__location").get_attribute("textContent")

                # Limpieza de precio para cumplir con el formato float [cite: 66]
                v_limpio = p_text.replace("$", "").replace(".", "").replace(",", "").replace("\n", "").strip()
                
                datos_lote.append({
                    "url": url,
                    "identificador": nombre.strip(),
                    "valor": float(v_limpio) if v_limpio.isdigit() else 0.0,
                    "caracteristicas": caract.strip(),
                    "direccion": loc.strip(),
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                })
            except:
                continue

        # GUARDADO CON UPSERT (Actualización de registros existentes) [cite: 72]
        if datos_lote:
            client = MongoClient("mongodb", 27017)
            db = client["Prueba"]
            coleccion = db["Pruebaaa"]

            for d in datos_lote:
                coleccion.update_one({"url": d["url"]}, {"$set": d}, upsert=True)
            print(f"✅ Página {pagina_actual} procesada y guardada.")

        # Avanzar a la siguiente página si no es la última del lote
        if i < CUANTAS_PAGINAS - 1:
            try:
                btn_sig = driver.find_element(By.CSS_SELECTOR, ".andes-pagination__button--next a")
                driver.execute_script("arguments[0].click();", btn_sig)
                time.sleep(5)
            except:
                print("🛑 No hay más resultados.")
                break

finally:
    if driver: driver.quit()
    print("\n🎉 Lote finalizado.")

🚀 Iniciando lote: Página 2 hasta 4

--- 📄 Extrayendo Página 2 ---
✅ Página 2 procesada y guardada.

--- 📄 Extrayendo Página 3 ---
✅ Página 3 procesada y guardada.

--- 📄 Extrayendo Página 4 ---
✅ Página 4 procesada y guardada.

🎉 Lote finalizado.


In [14]:
# ==============================================================================
# PROYECTO BIG DATA - GRUPO 2 (REAL ESTATE)
# Módulo de Extracción Profunda (Fase 1 y 2): Portal Inmobiliario
# ==============================================================================

import os
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

os.environ["DISPLAY"] = ":99"  
os.system("pkill -9 chrome && pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.* && rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Entorno virtual listo.")

options = Options()
# options.add_argument("--headless=new") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0")

# Configuración para scrapear por lotes (para dividirnos las páginas)
PAGINA_INICIAL = 1
CANTIDAD_PAGINAS = 3

# Listas de trabajo
catalogo_urls = []
datos_finales = []

try:
    driver = webdriver.Chrome(options=options)
    
    # ==============================================================================
    # FASE 1: RECOLECCIÓN EN EL CATÁLOGO
    # ==============================================================================
    desde = 1 if PAGINA_INICIAL == 1 else ((PAGINA_INICIAL - 1) * 48) + 1
    url_base = f"https://www.portalinmobiliario.com/arriendo/departamento/coquimbo-coquimbo_Desde_{desde}_NoIndex_True"
    driver.get(url_base)
    print(f"Iniciando lote: Página {PAGINA_INICIAL} hasta {PAGINA_INICIAL + CANTIDAD_PAGINAS - 1}")

    for nivel_pagina in range(CANTIDAD_PAGINAS):
        pagina_actual = PAGINA_INICIAL + i
        print(f"\n--- [FASE 1] Extrayendo Página {pagina_actual} ---")

        # Espera y Scroll Humano
        WebDriverWait(driver, 20).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ui-search-layout__item")))
        for _ in range(5):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(1)
        
        bloques = driver.find_elements(By.CSS_SELECTOR, ".ui-search-layout__item")
        datos_lote = []
        
        for bloque in bloques:
            try:
                # Datos base superficiales
                url = bloque.find_element(By.TAG_NAME, "a").get_attribute("href").split("#")[0].split("?")[0]
                titulo = bloque.find_element(By.CSS_SELECTOR, ".poly-component__title").get_attribute("textContent").strip()
                ubicacion_cruda = bloque.find_element(By.CSS_SELECTOR, ".poly-component__location").get_attribute("textContent").strip()
                ubi_minusc = ubicacion_cruda.lower()

                if "coquimbo" in ubi_minusc:
                    ubicacion = "Coquimbo"
                elif "la serena" in ubi_minusc:
                    ubicacion = "La Serena"
                else:
                    continue
                
                try:
                    p_text = bloque.find_element(By.CSS_SELECTOR, ".poly-price__current").get_attribute("textContent")
                except:
                    p_text = bloque.find_element(By.CSS_SELECTOR, ".poly-component__price").get_attribute("textContent")
                
                v_limpio = p_text.replace("$", "").replace(".", "").replace(",", "").replace("\n", "").strip()
                precio = float(v_limpio) if v_limpio.isdigit() else 0.0

                if url != "Sin URL" and precio > 0:
                    catalogo_urls.append({
                        "url": url,
                        "titulo": titulo,
                        "ubicacion": ubicacion,
                        "precio": precio
                    })
            except:
                continue

    print(f"✅ Catálogo listo: {len(catalogo_urls)} propiedades encontradas. Iniciando inspección profunda...")

    # ==============================================================================
    # FASE 2: INSPECCIÓN PROFUNDA POR PROPIEDAD
    # ==============================================================================
    for i, propiedad in enumerate(catalogo_urls):
        print(f"🔍 [{i+1}/{len(catalogo_urls)}] Inspeccionando: {propiedad['titulo'][:30]}...")
        
        # Diccionario base con la estructura exacta requerida
        registro = {
            "responsable": "Millaray Zalazar",
            "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
            "titulo": propiedad["titulo"],
            "ubicacion": propiedad["ubicacion"],
            "m2": 0,
            "precio": propiedad["precio"],
            "dormitorios": 0,
            "baños": 0,
            "estacionamiento": 0,
            "piscina": 0,
            "quincho": 0,
            "terraza": 0,
            "gimnasio": 0,
            "lavanderia": 0,
            "url": propiedad["url"] # Mantener la URL para el Upsert
        }

        try:
            driver.get(propiedad["url"])
            
            # Esperamos que cargue la tabla de características que descubriste
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, ".ui-pdp-collapsable__container"))
            )
            
            # Extraemos TODAS las filas de todas las tablas de la página
            filas_tabla = driver.find_elements(By.CSS_SELECTOR, ".andes-table__row")
            
            for fila in filas_tabla:
                # Convertimos el texto a minúsculas para facilitar la búsqueda
                # Usamos textContent porque a veces el botón "Ver más" oculta el texto
                texto_fila = fila.get_attribute("textContent").lower()
                
                # --- EXTRACCIÓN CON EXPRESIONES REGULARES (RegEx) ---
                if "superficie" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums: registro["m2"] = int(nums[0])
                        
                elif "dormitorios" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums: registro["dormitorios"] = int(nums[0])
                        
                elif "baños" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums: registro["baños"] = int(nums[0])
                        
                elif "estacionamiento" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums and int(nums[0]) > 0:
                        registro["estacionamiento"] = int(nums[0])
                    elif "sí" in texto_fila or "si" in texto_fila:
                        registro["estacionamiento"] = 1
                
                # --- EXTRACCIÓN BOOLEANA (1 o 0) ---
                elif "piscina" in texto_fila and "sí" in texto_fila:
                    registro["piscina"] = 1
                elif "quincho" in texto_fila and "sí" in texto_fila:
                    registro["quincho"] = 1
                elif "terraza" in texto_fila and "sí" in texto_fila:
                    registro["terraza"] = 1
                elif "gimnasio" in texto_fila and "sí" in texto_fila:
                    registro["gimnasio"] = 1
                elif "lavander" in texto_fila and "sí" in texto_fila: 
                    # "lavander" atrapa lavandería o lavadero
                    registro["lavanderia"] = 1

            datos_finales.append(registro)

        except Exception as e:
            print(f"⚠️ No se pudo leer la tabla de esta propiedad. Guardando con valores en 0.")
            datos_finales.append(registro)
            
        time.sleep(2) # Pausa vital para que Portal Inmobiliario no nos bloquee por navegar muy rápido

except Exception as e:
    print(f"🚨 Error crítico en Selenium: {e}")

finally:
    if driver is not None:
        driver.quit()
        print("\nNavegador cerrado.")

# ==============================================================================
# FASE 3: GUARDADO EN MONGODB (UPSERT)
# ==============================================================================
print("\n--- 💾 Iniciando proceso de guardado ---")
try:
    if datos_finales:
        client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
        db = client["Prueba"]
        coleccion = db["Pruebaaa"] 

        registros_nuevos = 0
        registros_actualizados = 0

        for d in datos_finales:
            resultado = coleccion.update_one(
                {"url": d["url"]}, 
                {"$set": d},       
                upsert=True        
            )

            if resultado.upserted_id:
                registros_nuevos += 1
            else:
                registros_actualizados += 1

        print(f"🎉 ¡Misión cumplida!")
        print(f"   -> {registros_nuevos} departamentos profundos guardados.")
        print(f"   -> {registros_actualizados} departamentos actualizados.")
    else:
        print("⚠️ No se encontró ningún dato para guardar.")

except Exception as e:
    print(f"🚨 Error conectando a la Base de Datos: {e}")

🧹 Entorno virtual listo.
Iniciando lote: Página 1 hasta 3

--- [FASE 1] Extrayendo Página 48 ---

--- [FASE 1] Extrayendo Página 48 ---

--- [FASE 1] Extrayendo Página 48 ---
✅ Catálogo listo: 144 propiedades encontradas. Iniciando inspección profunda...
🔍 [1/144] Inspeccionando: Arriendo Centro De Coquimbo: D...
🔍 [2/144] Inspeccionando: Se Arrienda Departamento Marzo...
🔍 [3/144] Inspeccionando: Edificio Manuel Jesús Rivera (...
🔍 [4/144] Inspeccionando: Arriendo Hermoso Penthouse A P...
🔍 [5/144] Inspeccionando: Arriendo Departamento Amoblado...
🔍 [6/144] Inspeccionando: Moderno Depto Amoblado Primer ...
🔍 [7/144] Inspeccionando: Depto Amoblado Primer Piso En ...
🔍 [8/144] Inspeccionando: Departamento Amoblado A Pasos ...
🔍 [9/144] Inspeccionando: Se Arrienda Departamento Amobl...
🔍 [10/144] Inspeccionando: Departamento En Arriendo, 3 Do...
🔍 [11/144] Inspeccionando: Nuevo, Amoblado, Año Corrido S...
🔍 [12/144] Inspeccionando: Departamento En Arriendo De 2d...
🔍 [13/144] Inspecciona

In [17]:
# ==============================================================================
# PROYECTO BIG DATA - GRUPO 2 (REAL ESTATE)
# Módulo de Extracción Profunda Distribuida: Portal Inmobiliario
# ==============================================================================

import os
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- CONFIGURACIÓN PARA ASIGNACIÓN DE PÁGINAS ---
PAGINA_INICIO = 3 
PAGINA_FIN = 3  
# ----------------------------------------------

os.environ["DISPLAY"] = ":99"  
os.system("pkill -9 chrome && pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.* && rm -rf /tmp/.org.chromium.Chromium.*")
print(f"🧹 Entorno virtual listo. Asignación: Páginas {PAGINA_INICIO} a la {PAGINA_FIN}.")

options = Options()
# options.add_argument("--headless=new") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0")

catalogo_urls = []
datos_finales = []

try:
    driver = webdriver.Chrome(options=options)
    
    # ==============================================================================
    # FASE 1: RECOLECCIÓN EN EL CATÁLOGO (CON PAGINACIÓN DINÁMICA)
    # ==============================================================================
    url_base = "https://www.portalinmobiliario.com/arriendo/departamento/coquimbo-coquimbo"

    # El ciclo ahora respeta los límites que definieron arriba
    for pagina_actual in range(PAGINA_INICIO, PAGINA_FIN + 1):
        print(f"\n--- 📄 [FASE 1] Recolectando enlaces de la Página {pagina_actual} ---")
        
        # Cálculo de salto de página de Portal Inmobiliario
        if pagina_actual == 1:
            url_pagina = url_base
        else:
            desde = ((pagina_actual - 1) * 48) + 1
            url_pagina = f"{url_base}_Desde_{desde}_NoIndex_True"
            
        driver.get(url_pagina)

        WebDriverWait(driver, 20).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ui-search-layout__item")))
        
        for _ in range(5):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2) 

        bloques = driver.find_elements(By.CSS_SELECTOR, ".ui-search-layout__item")
        
        for bloque in bloques:
            try:
                # 1. URL y Título
                url = bloque.find_element(By.TAG_NAME, "a").get_attribute("href").split("#")[0].split("?")[0]
                titulo = bloque.find_element(By.CSS_SELECTOR, ".poly-component__title").get_attribute("textContent").strip()
                
                # 2. Extracción y Normalización de Ubicación (Prioridad Coquimbo)
                ubicacion_cruda = bloque.find_element(By.CSS_SELECTOR, ".poly-component__location").get_attribute("textContent").strip()
                ubi_minuscula = ubicacion_cruda.lower()
                
                if "coquimbo" in ubi_minuscula:
                    ubicacion = "Coquimbo"      
                elif "la serena" in ubi_minuscula:
                    ubicacion = "La Serena"     
                else:
                    continue 

                # 3. Extracción de Precio
                try:
                    p_text = bloque.find_element(By.CSS_SELECTOR, ".poly-price__current").get_attribute("textContent")
                except:
                    p_text = bloque.find_element(By.CSS_SELECTOR, ".poly-component__price").get_attribute("textContent")
                
                v_limpio = p_text.replace("$", "").replace(".", "").replace(",", "").replace("\n", "").strip()
                precio = float(v_limpio) if v_limpio.isdigit() else 0.0

                if url != "Sin URL" and precio > 0:
                    catalogo_urls.append({
                        "url": url,
                        "titulo": titulo,
                        "ubicacion": ubicacion,
                        "precio": precio
                    })
            except:
                continue

    print(f"Catálogo listo: {len(catalogo_urls)} propiedades encontradas. Recolectando amenidades...")

    # ==============================================================================
    # FASE 2: INSPECCIÓN PROFUNDA POR PROPIEDAD
    # ==============================================================================
    for i, propiedad in enumerate(catalogo_urls):
        print(f"[{i+1}/{len(catalogo_urls)}] Inspeccionando: {propiedad['titulo'][:30]}...")
        
        registro = {
            "responsable": "Millaray Zalazar", # Aquí cambiar el nombre de cada uno
            "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
            "titulo": propiedad["titulo"],
            "ubicacion": propiedad["ubicacion"],
            "m2": 0,
            "precio": propiedad["precio"],
            "dormitorios": 0,
            "baños": 0,
            "estacionamiento": 0,
            "piscina": 0,
            "quincho": 0,
            "terraza": 0,
            "gimnasio": 0,
            "lavanderia": 0,
            "url": propiedad["url"] 
        }

        try:
            driver.get(propiedad["url"])
            
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, ".ui-pdp-collapsable__container"))
            )
            
            filas_tabla = driver.find_elements(By.CSS_SELECTOR, ".andes-table__row")
            
            for fila in filas_tabla:
                texto_fila = fila.get_attribute("textContent").lower()
                
                # Extracción Numérica
                if "superficie total" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums: registro["m2"] = int(nums[0])
                        
                elif "dormitorios" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums: registro["dormitorios"] = int(nums[0])
                        
                elif "baños" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums: registro["baños"] = int(nums[0])
                        
                elif "estacionamiento" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums and int(nums[0]) > 0:
                        registro["estacionamiento"] = int(nums[0])
                    elif "sí" in texto_fila or "si" in texto_fila:
                        registro["estacionamiento"] = 1
                
                # Extracción Booleana (1 o 0)
                elif "piscina" in texto_fila and "sí" in texto_fila:
                    registro["piscina"] = 1
                elif "quincho" in texto_fila and "sí" in texto_fila:
                    registro["quincho"] = 1
                elif "terraza" in texto_fila and "sí" in texto_fila:
                    registro["terraza"] = 1
                elif "gimnasio" in texto_fila and "sí" in texto_fila:
                    registro["gimnasio"] = 1
                elif "lavander" in texto_fila and "sí" in texto_fila: 
                    registro["lavanderia"] = 1

            datos_finales.append(registro)

        except Exception as e:
            print(f"Tabla no encontrada o página caída. Guardando con valores base.")
            datos_finales.append(registro)
            
        time.sleep(2) 

except Exception as e:
    print(f"Error crítico en Selenium: {e}")

finally:
    if driver is not None:
        driver.quit()
        print("\nNavegador cerrado.")

# ==============================================================================
# FASE 3: GUARDADO EN MONGODB (UPSERT)
# ==============================================================================
print("\n--- Proceso de guardado ---")
try:
    if datos_finales:
        client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
        db = client["Prueba"]
        coleccion = db["Pruebaaa"] 

        registros_nuevos = 0
        registros_actualizados = 0

        for d in datos_finales:
            resultado = coleccion.update_one(
                {"url": d["url"]}, 
                {"$set": d},       
                upsert=True        
            )

            if resultado.upserted_id:
                registros_nuevos += 1
            else:
                registros_actualizados += 1

        print(f"{datos_finales[0]['responsable']}")
        print(f"   -> {registros_nuevos} departamentos creados.")
        print(f"   -> {registros_actualizados} departamentos actualizados.")
    else:
        print("No se encontró ningún dato para guardar.")

except Exception as e:
    print(f"Error conectando a la Base de Datos: {e}")

🧹 Entorno virtual listo. Asignación: Páginas 3 a la 3.

--- 📄 [FASE 1] Recolectando enlaces de la Página 3 ---
Catálogo listo: 47 propiedades encontradas. Recolectando amenidades...
[1/47] Inspeccionando: Arriendo Departamento Amoblado...
[2/47] Inspeccionando: Arriendo Dpto. Año Corrido Cer...
[3/47] Inspeccionando: Se Arrienda Por Periodo De Mes...
[4/47] Inspeccionando: Departamento Dos Ambientes En ...
[5/47] Inspeccionando: Arriendo Departamento Año Corr...
[6/47] Inspeccionando: Depto 3d Alto Hacienda Arriend...
[7/47] Inspeccionando: Arriendo Departamento Año Corr...
[8/47] Inspeccionando: Se Arrienda Depto Año Corrido ...
[9/47] Inspeccionando: Sector La Herradura...
[10/47] Inspeccionando: Espectacular Depto En Peñuelas...
[11/47] Inspeccionando: Moderno Departamento En Marina...
[12/47] Inspeccionando: Departamento Con Finas Termina...
[13/47] Inspeccionando: Departamento Amoblado En Coqui...
[14/47] Inspeccionando: Arriendo Año Corrido Herradura...
[15/47] Inspeccionando: Ex

In [10]:
# ==============================================================================
# PROYECTO BIG DATA - GRUPO 2 (REAL ESTATE)
# Módulo de Extracción Profunda: GoPlaceIt (Arriendos en Coquimbo / La Serena)
# ==============================================================================

import os
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# --- 🎯 ASIGNACIÓN DE PÁGINAS PARA EL GRUPO ---
# Estudiante 1: Pone INICIO = 1, FIN = 5
# Estudiante 2: Pone INICIO = 6, FIN = 10
PAGINA_INICIO = 1  
PAGINA_FIN = 1   
# ----------------------------------------------

os.environ["DISPLAY"] = ":99"  
os.system("pkill -9 chrome && pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.* && rm -rf /tmp/.org.chromium.Chromium.*")
print(f"🧹 Entorno virtual listo. Asignación GoPlaceIt: Páginas {PAGINA_INICIO} a la {PAGINA_FIN}.")

options = Options()
# options.add_argument("--headless=new") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0")

catalogo_urls = []
datos_finales = []

try:
    driver = webdriver.Chrome(options=options)
    
    # ==============================================================================
    # FASE 1: RECOLECCIÓN EN EL CATÁLOGO (ARRIENDOS)
    # ==============================================================================
    # URL oficial de arriendos proporcionada
    url_base = "https://www.goplaceit.com/cl/departamento/arriendo/region/4-region-de-coquimbo"

    for pagina_actual in range(PAGINA_INICIO, PAGINA_FIN + 1):
        print(f"\n--- 📄 [FASE 1] Recolectando enlaces de la Página {pagina_actual} ---")
        
        # Paginación de GoPlaceIt
        url_pagina = f"{url_base}&page={pagina_actual}" if "?" in url_base else f"{url_base}?page={pagina_actual}"
        driver.get(url_pagina)

        try:
            # El Vigía: Esperamos por la clase estable 'featured card'
            WebDriverWait(driver, 15).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a[href*='/propiedad/']"))
            )
            
            
            # Scroll Humano
            for _ in range(6):
                driver.execute_script("window.scrollBy(0, 800);")
                time.sleep(1)
            time.sleep(2) 

            # Atrapamos los contenedores
            bloques = driver.find_elements(By.CSS_SELECTOR, "a[href*='/propiedad/']")
            
            for bloque in bloques:
                try:
                    url = bloque.get_attribute("href")
                    if not url or "javascript" in url: continue

                    # Leemos el texto de la tarjeta para filtrar por ubicación
                    texto_tarjeta = bloque.get_attribute("textContent").lower()
                    
                    if "coquimbo" in texto_tarjeta:
                        ubicacion = "Coquimbo"      
                    elif "la serena" in texto_tarjeta:
                        ubicacion = "La Serena"     
                    else:
                        continue 

                    catalogo_urls.append({
                        "url": url,
                        "ubicacion": ubicacion
                    })
                except:
                    continue

            print(f"✅ Se recolectaron enlaces en la página {pagina_actual}.")

        except TimeoutException:
            print("🚨 Advertencia: GoPlaceIt tardó en cargar. Posible fin de resultados.")
            continue
            
    print(f"\n✅ Catálogo listo: {len(catalogo_urls)} propiedades encontradas. Iniciando Fase 2...")

    # ==============================================================================
    # FASE 2: INSPECCIÓN PROFUNDA POR PROPIEDAD
    # ==============================================================================
    for i, propiedad in enumerate(catalogo_urls):
        # Latido del corazón: Imprime progreso cada 5 departamentos
        if (i + 1) % 5 == 0:
            print(f"⏳ Progreso: Inspeccionando propiedad {i + 1} de {len(catalogo_urls)}...")
        
        registro = {
            "responsable": "Millaray Zalazar", # Cambiar por el nombre del estudiante
            "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
            "titulo": "Propiedad GoPlaceIt",
            "ubicacion": propiedad["ubicacion"],
            "m2": 0,
            "precio": 0.0,
            "dormitorios": 0,
            "baños": 0,
            "estacionamiento": 0,
            "piscina": 0,
            "quincho": 0,
            "terraza": 0,
            "gimnasio": 0,
            "lavanderia": 0,
            "url": propiedad["url"] 
        }

        try:
            driver.get(propiedad["url"])
            
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            time.sleep(2) 
            
            # --- RASPADO MASIVO ---
            texto_total = driver.find_element(By.TAG_NAME, "body").get_attribute("textContent").lower()
            
            # Título
            try:
                registro["titulo"] = driver.find_element(By.TAG_NAME, "h1").get_attribute("textContent").strip()
            except: pass

            # --- EXTRACCIÓN DE PRECIO (Optimizado para Arriendos) ---
            try:
                # Buscamos el bloque de precio principal
                precio_bloque = driver.find_element(By.CSS_SELECTOR, ".gpi-featured-card-price-lg").get_attribute("textContent")
                # Limpiamos el texto, asumiendo formato en pesos (ej: $ 450.000)
                v_limpio = precio_bloque.lower().replace("$", "").replace("clp", "").replace(".", "").replace(",", "").replace("uf", "").strip()
                # Extraemos solo los primeros números seguidos que encontremos
                numeros_precio = re.findall(r'\d+', v_limpio)
                if numeros_precio:
                    registro["precio"] = float(numeros_precio[0])
            except:
                pass

            # --- EXTRACCIÓN NUMÉRICA (RegEx) ---
            match_m2 = re.search(r'(\d+)\s*(m2|m²|mts)', texto_total)
            if match_m2: registro["m2"] = int(match_m2.group(1))

            match_dorm = re.search(r'(\d+)\s*(habitacion|habitaciones|dormitorio)', texto_total)
            if match_dorm: registro["dormitorios"] = int(match_dorm.group(1))

            match_bano = re.search(r'(\d+)\s*baño', texto_total)
            if match_bano: registro["baños"] = int(match_bano.group(1))
            
            match_estac = re.search(r'(\d+)\s*estacionamiento', texto_total)
            if match_estac: 
                registro["estacionamiento"] = int(match_estac.group(1))
            elif "estacionamiento" in texto_total:
                registro["estacionamiento"] = 1

            # --- AMENIDADES BOOLEANAS ---
            if "piscina" in texto_total: registro["piscina"] = 1
            if "quincho" in texto_total: registro["quincho"] = 1
            if "terraza" in texto_total: registro["terraza"] = 1
            if "gimnasio" in texto_total: registro["gimnasio"] = 1
            if "lavander" in texto_total: registro["lavanderia"] = 1

            datos_finales.append(registro)

        except Exception as e:
            # Si falla la lectura profunda, guardamos con valores base para no perder la URL
            datos_finales.append(registro)
            
        time.sleep(2) 

except Exception as e:
    print(f"🚨 Error crítico en Selenium: {e}")

finally:
    if driver is not None: driver.quit()

# ==============================================================================
# GUARDADO EN MONGODB (UPSERT)
# ==============================================================================
print("\n--- 💾 Iniciando guardado en Base de Datos ---")
try:
    if datos_finales:
        # Asegúrate de usar localhost para evitar el error de conexión
        client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
        db = client["Prueba"]
        coleccion = db["Pruebaaa"] 

        registros_nuevos, registros_actualizados = 0, 0

        for d in datos_finales:
            resultado = coleccion.update_one({"url": d["url"]}, {"$set": d}, upsert=True)
            if resultado.upserted_id: registros_nuevos += 1
            else: registros_actualizados += 1

        print(f"🎉 ¡Extracción GoPlaceIt Arriendos Completada, {datos_finales[0]['responsable']}!")
        print(f"   -> {registros_nuevos} propiedades nuevas.")
        print(f"   -> {registros_actualizados} propiedades actualizadas.")
    else:
        print("⚠️ No se extrajeron datos válidos.")
except Exception as e:
    print(f"🚨 Error de Base de Datos: {e}")

🧹 Entorno virtual listo. Asignación GoPlaceIt: Páginas 1 a la 1.

--- 📄 [FASE 1] Recolectando enlaces de la Página 1 ---
✅ Se recolectaron enlaces en la página 1.

✅ Catálogo listo: 24 propiedades encontradas. Iniciando Fase 2...
⏳ Progreso: Inspeccionando propiedad 5 de 24...
⏳ Progreso: Inspeccionando propiedad 10 de 24...
⏳ Progreso: Inspeccionando propiedad 15 de 24...
⏳ Progreso: Inspeccionando propiedad 20 de 24...

--- 💾 Iniciando guardado en Base de Datos ---
🎉 ¡Extracción GoPlaceIt Arriendos Completada, Millaray Zalazar!
   -> 0 propiedades nuevas.
   -> 24 propiedades actualizadas.


In [27]:
# ==============================================================================
# PROYECTO BIG DATA - GRUPO 2 (REAL ESTATE)
# Módulo de Extracción Ultra-Rápida con Desencriptador Base64: Mitula (Arriendos)
# ==============================================================================

import os
import time
import re
import base64  # 👈 Herramienta clave para romper la seguridad de Mitula
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# --- 🎯 ASIGNACIÓN DE PÁGINAS PARA EL GRUPO ---
PAGINA_INICIO = 1  
PAGINA_FIN = 1   
# ----------------------------------------------

os.environ["DISPLAY"] = ":99"  
os.system("pkill -9 chrome && pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.* && rm -rf /tmp/.org.chromium.Chromium.*")
print(f"🧹 Entorno virtual listo. Asignación Mitula: Páginas {PAGINA_INICIO} a la {PAGINA_FIN}.")

options = Options()
# options.add_argument("--headless=new") # Comentado para ver la acción en vivo
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0")

datos_finales = []

try:
    driver = webdriver.Chrome(options=options)
    
    # URL oficial de Mitula
    url_base = "https://casas.mitula.cl/find?operationType=rent&propertyType=apartment&geoId=R231672"

    for pagina_actual in range(PAGINA_INICIO, PAGINA_FIN + 1):
        print(f"\n--- 📄 Procesando Página {pagina_actual} ---")
        
        url_pagina = f"{url_base}&page={pagina_actual}"
        driver.get(url_pagina)

        try:
            # 1. El Vigía: Esperamos por la clase principal de la tarjeta
            WebDriverWait(driver, 15).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "article.listing-card"))
            )
            
            # Scroll Humano rápido para renderizar imágenes y cargar elementos
            for _ in range(4):
                driver.execute_script("window.scrollBy(0, 800);")
                time.sleep(1)
            time.sleep(1) 

            # 2. Atrapamos todas las tarjetas de la página
            bloques = driver.find_elements(By.CSS_SELECTOR, "article.listing-card")
            print(f"🔍 Analizando {len(bloques)} departamentos en esta página...")
            
            for bloque in bloques:
                try:
                    # --- FILTRO DE UBICACIÓN A PRUEBA DE BALAS ---
                    texto_tarjeta = bloque.get_attribute("textContent").lower()
                    
                    if "coquimbo" in texto_tarjeta:
                        ubicacion = "Coquimbo"      
                    elif "la serena" in texto_tarjeta:
                        ubicacion = "La Serena"     
                    else:
                        print("   -> ❌ Rechazado: Ubicación no coincide (es de otra ciudad)")
                        continue 

                    # --- URL DE LA PROPIEDAD (DESENCRIPTADOR BASE64) ---
                    url = "Sin URL"
                    
                    try:
                        # 1. Buscamos el código secreto de Mitula
                        codigo_secreto = bloque.get_attribute("data-clickdestination")
                        
                        if codigo_secreto:
                            # 2. Rompemos la encriptación Base64
                            url_decodificada = base64.b64decode(codigo_secreto).decode('utf-8', errors='ignore')
                            
                            # 3. Le pegamos el dominio principal
                            url = "https://casas.mitula.cl" + url_decodificada
                        else:
                            # Plan B: Si no hay código secreto, usamos el ID único de la tarjeta
                            id_unico = bloque.get_attribute("data-listingid")
                            if id_unico:
                                url = f"https://casas.mitula.cl/propiedad_id_{id_unico}"
                    except Exception as e:
                        pass

                    if url == "Sin URL":
                        print("   -> ❌ Rechazado: Tarjeta completamente vacía o sin identificador.")
                        continue

                    # --- EXTRACCIÓN DE PRECIO ---
                    try:
                        precio_texto = bloque.find_element(By.CSS_SELECTOR, ".price__actual").get_attribute("textContent")
                        v_limpio = precio_texto.replace("$", "").replace("/mes", "").replace(".", "").replace(",", "").strip()
                        precio = float(v_limpio) if v_limpio.isdigit() else 0.0
                    except:
                        precio = 0.0
                        
                    if precio == 0.0: 
                        print("   -> ❌ Rechazado: El precio está oculto o no es numérico")
                        continue 

                    # --- EXTRACCIÓN DE CARACTERÍSTICAS Y AMENIDADES ---
                    try:
                        props_texto = bloque.find_element(By.CSS_SELECTOR, ".listing-card__properties").get_attribute("textContent")
                    except: props_texto = ""
                    
                    try:
                        facil_texto = bloque.find_element(By.CSS_SELECTOR, ".listing-card__facilities").get_attribute("textContent")
                    except: facil_texto = ""

                    texto_total = f"{props_texto} {facil_texto}".lower()

                    m2, dormitorios, banos = 0, 0, 0

                    match_m2 = re.search(r'(\d+)\s*(m2|m²|mts)', texto_total)
                    if match_m2: m2 = int(match_m2.group(1))

                    match_dorm = re.search(r'(\d+)\s*dormitorio', texto_total)
                    if match_dorm: dormitorios = int(match_dorm.group(1))

                    match_bano = re.search(r'(\d+)\s*baño', texto_total)
                    if match_bano: banos = int(match_bano.group(1))

                    # --- EMPAQUETADO ---
                    datos_finales.append({
                        "responsable": "Millaray Zalazar", # 👈 Asegúrate de que tu grupo cambie esto por sus nombres
                        "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
                        "titulo": "Departamento Mitula", 
                        "ubicacion": ubicacion,
                        "m2": m2,
                        "precio": precio,
                        "dormitorios": dormitorios,
                        "baños": banos,
                        "estacionamiento": 1 if "estacionamiento" in texto_total else 0,
                        "piscina": 1 if "piscina" in texto_total else 0,
                        "quincho": 1 if "quincho" in texto_total else 0,
                        "terraza": 1 if "terraza" in texto_total else 0,
                        "gimnasio": 1 if "gimnasio" in texto_total else 0,
                        "lavanderia": 1 if "lavander" in texto_total else 0,
                        "url": url
                    })
                    print(f"   -> ✅ Capturado: Depto en {ubicacion} por ${precio}")

                except Exception as e:
                    print(f"   -> ⚠️ Error inesperado al leer bloque: {e}")
                    continue 

            print(f"✅ Análisis de página {pagina_actual} completado.")

        except TimeoutException:
            print("🚨 Advertencia: Mitula tardó en cargar o no hay más páginas.")
            continue
            
        time.sleep(2) 

except Exception as e:
    print(f"🚨 Error crítico en Selenium: {e}")

finally:
    if driver is not None: driver.quit()

# ==============================================================================
# GUARDADO EN MONGODB (UPSERT)
# ==============================================================================
print("\n--- 💾 Iniciando guardado en Base de Datos ---")
try:
    if datos_finales:
        # Usando localhost para tu entorno
        client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
        db = client["Prueba"]
        coleccion = db["Pruebaaa"] 

        registros_nuevos, registros_actualizados = 0, 0

        for d in datos_finales:
            resultado = coleccion.update_one({"url": d["url"]}, {"$set": d}, upsert=True)
            if resultado.upserted_id: registros_nuevos += 1
            else: registros_actualizados += 1

        print(f"🎉 ¡Extracción Mitula Completada, {datos_finales[0]['responsable']}!")
        print(f"   -> {registros_nuevos} propiedades nuevas guardadas en Mongo.")
        print(f"   -> {registros_actualizados} propiedades actualizadas.")
    else:
        print("⚠️ No se extrajeron datos válidos para guardar.")
except Exception as e:
    print(f"🚨 Error de Base de Datos: {e}")

🧹 Entorno virtual listo. Asignación Mitula: Páginas 1 a la 1.

--- 📄 Procesando Página 1 ---


KeyboardInterrupt: 

In [31]:
# ==============================================================================
# Módulo de Extracción Profunda y Anti-Duplicados: Mitula (Arriendos)
# ==============================================================================

import os
import time
import re
import base64  
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# --- CONFIGURACIÓN PARA ASIGNACIÓN DE PÁGINAS---
PAGINA_INICIO = 1  
PAGINA_FIN = 1  
# ----------------------------------------------

os.environ["DISPLAY"] = ":99"  
os.system("pkill -9 chrome && pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.* && rm -rf /tmp/.org.chromium.Chromium.*")
print(f"🧹 Entorno virtual listo. Asignación Mitula: Páginas {PAGINA_INICIO} a la {PAGINA_FIN}.")

options = Options()
# options.add_argument("--headless=new") # Comentado para que veas la magia
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0")

catalogo_urls = []
datos_finales = []

try:
    driver = webdriver.Chrome(options=options)
    url_base = "https://casas.mitula.cl/find?operationType=rent&propertyType=apartment&geoId=R231672"

    # ==============================================================================
    # FASE 1: RECOLECCIÓN EN EL CATÁLOGO (PRECIOS, IDs Y ENLACES)
    # ==============================================================================
    for pagina_actual in range(PAGINA_INICIO, PAGINA_FIN + 1):
        print(f"\n--- [FASE 1] Procesando Página {pagina_actual} ---")
        
        url_pagina = f"{url_base}&page={pagina_actual}"
        driver.get(url_pagina)

        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "article.listing-card"))
            )
            
            for _ in range(4):
                driver.execute_script("window.scrollBy(0, 800);")
                time.sleep(1)
            time.sleep(1) 

            bloques = driver.find_elements(By.CSS_SELECTOR, "article.listing-card")
            
            for bloque in bloques:
                try:
                    # 1. LA LLAVE MAESTRA: El ID único del departamento
                    listing_id = bloque.get_attribute("data-listingid")
                    if not listing_id: continue # Si no tiene ID, lo ignoramos por seguridad
                    
                    # 2. Filtro de Ubicación
                    texto_tarjeta = bloque.get_attribute("textContent").lower()
                    if "coquimbo" in texto_tarjeta: ubicacion = "Coquimbo"      
                    elif "la serena" in texto_tarjeta: ubicacion = "La Serena"     
                    else: continue 

                    # 3. Desencriptador de Enlaces Base64
                    url = "Sin URL"
                    try:
                        codigo_secreto = bloque.get_attribute("data-clickdestination")
                        if codigo_secreto:
                            url_decodificada = base64.b64decode(codigo_secreto).decode('utf-8', errors='ignore')
                            url = "https://casas.mitula.cl" + url_decodificada
                        else:
                            url = f"https://casas.mitula.cl/propiedad_id_{listing_id}"
                    except: pass

                    if url == "Sin URL" or "javascript" in url: continue

                    # 4. Extracción de Precio Base
                    try:
                        precio_texto = bloque.find_element(By.CSS_SELECTOR, ".price__actual").get_attribute("textContent")
                        v_limpio = precio_texto.replace("$", "").replace("/mes", "").replace(".", "").replace(",", "").strip()
                        precio = float(v_limpio) if v_limpio.isdigit() else 0.0
                    except: precio = 0.0
                        
                    if precio == 0.0: continue 

                    # 5. Extracción Numérica Base
                    m2, dormitorios, banos = 0, 0, 0
                    match_m2 = re.search(r'(\d+)\s*(m2|m²|mts)', texto_tarjeta)
                    if match_m2: m2 = int(match_m2.group(1))

                    match_dorm = re.search(r'(\d+)\s*dormitorio', texto_tarjeta)
                    if match_dorm: dormitorios = int(match_dorm.group(1))

                    match_bano = re.search(r'(\d+)\s*baño', texto_tarjeta)
                    if match_bano: banos = int(match_bano.group(1))

                    # Guardamos en la sala de espera
                    catalogo_urls.append({
                        "responsable": "Millaray Zalazar", # Aqui ponen su nombre
                        "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
                        "listing_id": listing_id, # GUARDAMOS EL ID
                        "titulo": "Departamento Mitula", 
                        "ubicacion": ubicacion,
                        "m2": m2,
                        "precio": precio,
                        "dormitorios": dormitorios,
                        "baños": banos,
                        "url": url
                    })

                except Exception as e:
                    continue 

            print(f"✅ Se recolectaron enlaces base de la página {pagina_actual}.")

        except TimeoutException:
            print("Advertencia: Mitula tardó en cargar. Posible fin de resultados.")
            continue
            
    print(f"\nCatálogo listo: {len(catalogo_urls)} propiedades encontradas. Iniciando Fase 2...")

    # ==============================================================================
    # FASE 2: INSPECCIÓN PROFUNDA (Buscando amenidades ocultas)
    # ==============================================================================
    for i, propiedad in enumerate(catalogo_urls):
        if (i + 1) % 5 == 0:
            print(f"Progreso: Buscando en la propiedad {i + 1} de {len(catalogo_urls)}...")
        
        # Inicializamos en 0
        propiedad["estacionamiento"] = 0
        propiedad["piscina"] = 0
        propiedad["quincho"] = 0
        propiedad["terraza"] = 0
        propiedad["gimnasio"] = 0
        propiedad["lavanderia"] = 0

        try:
            driver.get(propiedad["url"])
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            time.sleep(1.5) 
            
            # Raspado Masivo del Body (Incluye descripción completa)
            texto_total = driver.find_element(By.TAG_NAME, "body").get_attribute("textContent").lower()
            
            if "estacionamiento" in texto_total: propiedad["estacionamiento"] = 1
            if "piscina" in texto_total: propiedad["piscina"] = 1
            if "quincho" in texto_total: propiedad["quincho"] = 1
            if "terraza" in texto_total: propiedad["terraza"] = 1
            if "gimnasio" in texto_total: propiedad["gimnasio"] = 1
            if "lavander" in texto_total: propiedad["lavanderia"] = 1

        except Exception as e:
            pass # Si falla, guarda la propiedad con amenidades en 0
            
        datos_finales.append(propiedad)

except Exception as e:
    print(f"Error crítico en Selenium: {e}")

finally:
    if driver is not None: driver.quit()

# ==============================================================================
# GUARDADO EN MONGODB (UPSERT CON LISTING_ID)
# ==============================================================================
print("\n--- Iniciando guardado en Base de Datos ---")
try:
    if datos_finales:
        client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
        db = client["Prueba"]
        coleccion = db["Pruebaaa"] 

        registros_nuevos, registros_actualizados = 0, 0

        for d in datos_finales:
            # LA MAGIA ANTI-DUPLICADOS: Buscamos por listing_id, no por URL
            resultado = coleccion.update_one(
                {"listing_id": d["listing_id"]}, 
                {"$set": d}, 
                upsert=True
            )
            
            if resultado.upserted_id: registros_nuevos += 1
            else: registros_actualizados += 1

        print(f"¡Extracción Completada, {datos_finales[0]['responsable']}!")
        print(f"   -> {registros_nuevos} propiedades NUEVAS guardadas.")
        print(f"   -> {registros_actualizados} propiedades ACTUALIZADAS (sin duplicar).")
    else:
        print("No se extrajeron datos válidos.")
except Exception as e:
    print(f"🚨 Error de Base de Datos: {e}")

🧹 Entorno virtual listo. Asignación Mitula: Páginas 1 a la 1.

--- 📄 [FASE 1] Procesando Página 1 ---
✅ Se recolectaron enlaces base de la página 1.

✅ Catálogo listo: 28 propiedades encontradas. Iniciando Fase 2...
⏳ Progreso: Buceando en la propiedad 5 de 28...
⏳ Progreso: Buceando en la propiedad 10 de 28...
⏳ Progreso: Buceando en la propiedad 15 de 28...
⏳ Progreso: Buceando en la propiedad 20 de 28...
⏳ Progreso: Buceando en la propiedad 25 de 28...

--- 💾 Iniciando guardado en Base de Datos ---
🎉 ¡Extracción Mitula Completada, Millaray Zalazar!
   -> 0 propiedades NUEVAS guardadas.
   -> 28 propiedades ACTUALIZADAS (sin duplicar).
